# E5 — Análisis de variables y preprocesamiento

Este notebook responde las cuatro preguntas a partir de `dataset/data_messi_full.csv`. Está hecho sólo con la biblioteca estándar de Python para que pueda ejecutarse aun si `pandas` no está instalado.

In [ ]:
from pathlib import Path
from collections import Counter
import csv

# Permite ejecutar el notebook desde la raíz del repositorio o desde E5/.
candidates = [
    Path('dataset/data_messi_full.csv'),
    Path('E5/dataset/data_messi_full.csv'),
]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('No se encontró data_messi_full.csv')

with data_path.open(encoding='utf-8-sig', newline='') as file:
    rows = list(csv.DictReader(file))

print(f'Archivo: {data_path}')
print(f'Filas: {len(rows)} | Columnas: {len(rows[0])}')

## 1. Inconsistencias en `Playing_Position`

Primero se muestran los valores con `repr`, que hace visibles los espacios finales. Luego se aplica una limpieza mínima con `strip()` (y `upper()` como normalización defensiva).

In [ ]:
raw_positions = Counter(row['Playing_Position'] for row in rows)
print('Antes de limpiar:')
for value, count in sorted(raw_positions.items()):
    print(f'{value!r}: {count}')

clean_positions = [row['Playing_Position'].strip().upper() for row in rows]
clean_position_counts = Counter(clean_positions)

print('\nDespués de limpiar:')
for value, count in sorted(clean_position_counts.items()):
    print(f'{value!r}: {count}')

Los espacios hacen que, por ejemplo, `CF` y `CF ` sean categorías distintas. Una codificación aplicada antes de limpiar conservaría ese error y crearía columnas separadas.

**Respuesta correcta: Hacer una limpieza de la columna y luego codificar.**

## 2. Uso de `Opponent` como variable predictora

Se revisa su cardinalidad y cuántas categorías aparecen una sola vez.

In [ ]:
opponent_counts = Counter(row['Opponent'].strip() for row in rows)
n_opponents = len(opponent_counts)
singletons = sum(count == 1 for count in opponent_counts.values())

print(f'Rivales distintos: {n_opponents}')
print(f'Rivales que aparecen una sola vez: {singletons}')
print(f'Proporción categorías/filas: {n_opponents / len(rows):.2%}')
print('Más frecuentes:', opponent_counts.most_common(10))

# Ejemplo de Frequency Encoding: frecuencia relativa de cada rival.
opponent_frequency = {
    opponent: count / len(rows)
    for opponent, count in opponent_counts.items()
}
opponent_encoded = [
    opponent_frequency[row['Opponent'].strip()]
    for row in rows
]
print('Primeros valores codificados:', opponent_encoded[:5])

`Opponent` tiene 98 categorías en sólo 704 observaciones; un one-hot encoding produciría muchas columnas, varias de ellas muy poco frecuentes. Frequency Encoding mantiene una sola columna. Target Encoding también es válido, pero debe ajustarse únicamente con los datos de entrenamiento y, preferentemente, dentro de cada fold para evitar fuga de información.

**Respuesta correcta: Frequency Encoding o Target Encoding.**

## 3. Tramo del partido en el que convierte más goles

Para comparar tramos con la misma duración se discretiza `Minute_num` en intervalos temporales de 15 minutos. Se mantienen intervalos adicionales para el tiempo suplementario.

In [ ]:
minutes = [int(row['Minute_num']) for row in rows if row['Minute_num'].strip()]
interval_width = 15

def time_interval(minute, width=interval_width):
    lower = ((minute - 1) // width) * width + 1
    upper = lower + width - 1
    return f'{lower}-{upper}'

minute_intervals = Counter(time_interval(minute) for minute in minutes)
ordered_intervals = sorted(
    minute_intervals.items(),
    key=lambda item: int(item[0].split('-')[0]),
)

print(f'Mínimo: {min(minutes)} | Máximo: {max(minutes)}')
for interval, count in ordered_intervals:
    print(f'{interval} min: {count} goles')

most_goals_interval = max(ordered_intervals, key=lambda item: item[1])
print(f'\nTramo con más goles: {most_goals_interval[0]} minutos '
      f'({most_goals_interval[1]} goles)')

La discretización por igual frecuencia forzaría cantidades similares de goles en cada grupo y sus tramos tendrían duraciones diferentes, por lo que no respondería bien la pregunta temporal. En estos datos, el intervalo de **76 a 90 minutos** concentra la mayor cantidad: **143 goles**.

**Respuesta correcta: Discretizar en intervalos de igual longitud de tiempo.**

## 4. Análisis de la variable objetivo `Has_Assist`

In [ ]:
target_raw = [row['Has_Assist'].strip() for row in rows]
missing_target = sum(value == '' for value in target_raw)
target = [int(value) for value in target_raw if value != '']
target_counts = Counter(target)

print(f'Nulos/vacíos: {missing_target}')
for value in sorted(target_counts):
    count = target_counts[value]
    print(f'{value}: {count} ({count / len(target):.2%})')

No hay valores nulos. La clase 1 representa 490 de 704 casos (**69,60 %**) y la clase 0, 214 (**30,40 %**). Hay una diferencia entre clases, pero ninguna alcanza los extremos propuestos de más del 80 % o menos del 10 %.

**Respuesta correcta: El dataset está ligeramente desbalanceado.**

## Resumen de respuestas

1. **Hacer una limpieza de la columna y luego codificar.**
2. **Frequency Encoding o Target Encoding.**
3. **Discretizar en intervalos de igual longitud de tiempo.**
4. **El dataset está ligeramente desbalanceado.**